# Gold — Fact Monthly Economics (Kimball Periodic Snapshot)

Joins Silver tables and unpivots into a narrow periodic snapshot fact table.
Follows Kimball Data Warehouse Toolkit conventions — one row per month per indicator, with FK references to all four dimensions.

**Sources:** `silver.exchange_rates`, `silver.central_bank_indicators`, `silver.gdp_indicators`  
**Dimensions:** `gold.dim_date`, `gold.dim_indicator`, `gold.dim_source`, `gold.dim_geography`  
**Output:** `gold.fact_monthly_economics` (Delta table)  
**Grain:** One row per calendar month per economic indicator  

| Column | Type | Description |
|---|---|---|
| `month_key` | int | Grain surrogate — YYYYMM (e.g. 202401) |
| `date_key` | int | FK → `gold.dim_date` — YYYYMMDD of first day of month |
| `indicator_key` | int | FK → `gold.dim_indicator` |
| `source_key` | int | FK → `gold.dim_source` — denormalised from dim_indicator |
| `geography_key` | int | FK → `gold.dim_geography` — 1 (Iceland) for all rows |
| `value` | double | Indicator value for the month |
| `is_estimated` | boolean | True if value is provisional or estimated |
| `row_created_at` | timestamp | When this row was first inserted into the warehouse |
| `row_updated_at` | timestamp | When this row was last updated by the pipeline |

In [ ]:
df_fx  = spark.read.format("delta").table("silver.exchange_rates")
df_cb  = spark.read.format("delta").table("silver.central_bank_indicators")
df_gdp = spark.read.format("delta").table("silver.gdp_indicators")
df_ind = spark.read.format("delta").table("gold.dim_indicator")

print(f"Exchange rates:  {df_fx.count()} rows")
print(f"Central bank:    {df_cb.count()} rows")
print(f"GDP:             {df_gdp.count()} rows")
print(f"Dim indicator:   {df_ind.count()} rows")

In [ ]:
df_fx.createOrReplaceTempView("v_exchange_rates")
df_cb.createOrReplaceTempView("v_central_bank_indicators")
df_gdp.createOrReplaceTempView("v_gdp_indicators")
df_ind.createOrReplaceTempView("v_dim_indicator")

In [ ]:
df_gold = spark.sql("""
    -- Monthly averages from daily FX and equity data
    WITH fx_monthly AS (
        SELECT
            YEAR(date)                   AS year,
            MONTH(date)                  AS month,
            ROUND(AVG(iskusd_close), 6)  AS avg_iskusd,
            ROUND(AVG(iskeur_close), 6)  AS avg_iskeur,
            ROUND(AVG(eurusd_close), 6)  AS avg_eurusd,
            ROUND(AVG(omx_close),    2)  AS avg_omx
        FROM v_exchange_rates
        GROUP BY YEAR(date), MONTH(date)
    ),

    -- End-of-month policy rate using ROW_NUMBER to take the last reading per month
    policy AS (
        SELECT year, month, ROUND(value, 4) AS policy_rate
        FROM (
            SELECT
                YEAR(date)  AS year,
                MONTH(date) AS month,
                value,
                ROW_NUMBER() OVER (
                    PARTITION BY YEAR(date), MONTH(date)
                    ORDER BY date DESC
                ) AS rn
            FROM v_central_bank_indicators
            WHERE metric = 'policy_rate'
        )
        WHERE rn = 1
    ),

    -- Monthly CPI reading
    cpi AS (
        SELECT
            YEAR(date)          AS year,
            MONTH(date)         AS month,
            ROUND(value, 4)     AS cpi
        FROM v_central_bank_indicators
        WHERE metric = 'cpi'
    ),

    -- Quarterly GDP — joined to months via quarter mapping
    gdp AS (
        SELECT year, q, ROUND(value, 2) AS gdp_yoy_growth
        FROM v_gdp_indicators
    ),

    -- Wide monthly snapshot joining all three sources
    combined AS (
        SELECT
            fx.year * 100 + fx.month              AS month_key,
            fx.year * 10000 + fx.month * 100 + 1  AS date_key,
            fx.avg_iskusd,
            fx.avg_iskeur,
            fx.avg_eurusd,
            fx.avg_omx,
            p.policy_rate,
            c.cpi,
            g.gdp_yoy_growth
        FROM fx_monthly fx
        LEFT JOIN policy p ON fx.year = p.year AND fx.month = p.month
        LEFT JOIN cpi    c ON fx.year = c.year AND fx.month = c.month
        LEFT JOIN gdp    g ON fx.year = g.year
            AND CASE
                WHEN fx.month IN (1,2,3) THEN 1
                WHEN fx.month IN (4,5,6) THEN 2
                WHEN fx.month IN (7,8,9) THEN 3
                ELSE 4
            END = g.q
    ),

    -- Unpivot wide to narrow: one row per month per indicator
    -- indicator_key values correspond to gold.dim_indicator surrogate keys
    unpivoted AS (
        SELECT month_key, date_key, 1 AS indicator_key, avg_iskusd     AS value FROM combined WHERE avg_iskusd     IS NOT NULL
        UNION ALL
        SELECT month_key, date_key, 2 AS indicator_key, avg_iskeur     AS value FROM combined WHERE avg_iskeur     IS NOT NULL
        UNION ALL
        SELECT month_key, date_key, 3 AS indicator_key, avg_eurusd     AS value FROM combined WHERE avg_eurusd     IS NOT NULL
        UNION ALL
        SELECT month_key, date_key, 4 AS indicator_key, avg_omx        AS value FROM combined WHERE avg_omx        IS NOT NULL
        UNION ALL
        SELECT month_key, date_key, 5 AS indicator_key, policy_rate    AS value FROM combined WHERE policy_rate    IS NOT NULL
        UNION ALL
        SELECT month_key, date_key, 6 AS indicator_key, cpi            AS value FROM combined WHERE cpi            IS NOT NULL
        UNION ALL
        SELECT month_key, date_key, 7 AS indicator_key, gdp_yoy_growth AS value FROM combined WHERE gdp_yoy_growth IS NOT NULL
    )

    -- Final fact: attach dimension FKs and audit columns
    SELECT
        u.month_key,
        u.date_key,
        u.indicator_key,
        i.source_key,
        1                   AS geography_key,
        u.value,
        FALSE               AS is_estimated,
        CURRENT_TIMESTAMP() AS row_created_at,
        CURRENT_TIMESTAMP() AS row_updated_at
    FROM unpivoted u
    JOIN v_dim_indicator i ON u.indicator_key = i.indicator_key
    ORDER BY u.month_key, u.indicator_key
""")

if df_gold.isEmpty():
    raise ValueError("[gold_fact_monthly_economics] No rows returned - halting.")

df_gold.show(10)

In [ ]:
df_gold.createOrReplaceTempView("v_fact_monthly_economics")

if not spark.catalog.tableExists("gold.fact_monthly_economics"):
    df_gold.write.format("delta").saveAsTable("gold.fact_monthly_economics")
else:
    spark.sql("""
        MERGE INTO gold.fact_monthly_economics AS target
        USING v_fact_monthly_economics AS source
        ON target.month_key = source.month_key
        AND target.indicator_key = source.indicator_key
        WHEN MATCHED THEN UPDATE SET
            target.date_key      = source.date_key,
            target.source_key    = source.source_key,
            target.geography_key = source.geography_key,
            target.value         = source.value,
            target.is_estimated  = source.is_estimated,
            target.row_updated_at = source.row_updated_at
        WHEN NOT MATCHED THEN INSERT (
            month_key, date_key, indicator_key, source_key, geography_key,
            value, is_estimated, row_created_at, row_updated_at
        )
        VALUES (
            source.month_key, source.date_key, source.indicator_key, source.source_key,
            source.geography_key, source.value, source.is_estimated,
            source.row_created_at, source.row_updated_at
        )
    """)

print(f"Saved to gold.fact_monthly_economics - {spark.table('gold.fact_monthly_economics').count()} rows")